# J1 — bout en bout, avant toute théorie

**Séance 1.** On ne vous explique rien de ce qui se passe à l'intérieur du modèle.
Regardez ce qui **entre** et ce qui **sort**.

> **La règle de la séance :** une soumission déposée vaut mieux qu'un modèle parfait qui n'existe
> pas. Un score sans référence ne veut rien dire — d'où la cellule 5.

In [ ]:
# 0 — les bibliothèques du cours, et les versions pour lesquelles il a été écrit :
#        pandas 3.0.6 · numpy 2.4.3 · scikit-learn 1.9.0 · jinja2
#
# Le code tourne AUSSI sous pandas 2 (c'est ce que Kaggle fournit) : les deux sont testés.
#
# SUR KAGGLE : rien à faire. Les bibliothèques sont déjà installées et l'INTERNET EST COUPÉ :
# une commande d'installation y échoue. Ce n'est pas une erreur de votre part.
#
# EN LOCAL, une seule fois — dans un terminal :
#     pip install "pandas==3.0.6" "numpy==2.4.3" "scikit-learn==1.9.0" "jinja2"
# dans une cellule d'un notebook local, la même chose avec un point d'exclamation devant.

# Ce que vous avez, à cet instant :
import importlib

for nom in ("pandas", "numpy", "sklearn", "jinja2"):
    try:
        print(f"{nom:<8}", importlib.import_module(nom).__version__)
    except ImportError:
        print(f"{nom:<8} absent — voir les lignes d'installation ci-dessus")

In [ ]:
# 1 — les outils
import glob
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

In [ ]:
# 2 — où sont les fichiers (le notebook marche sur Kaggle ET en local)
# On CHERCHE train.csv au lieu de supposer un chemin. Kaggle a déjà changé ses points de
# montage (/kaggle/input/<identifiant> puis /kaggle/input/competitions/<identifiant>) : un
# chemin recopié à la main se périme, une recherche non.
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie", os.path.join("..", "sortie"),
                            os.path.join("..", "Donnees_competition", "sortie")])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)

In [ ]:
# 3 — lire, séparer la cible
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

# Ce qu'on met de côté aujourd'hui, sans l'expliquer : les identifiants, les colonnes
# connues APRÈS l'issue (elles n'existeraient pas au moment de décider), les dates brutes
# et les colonnes saisies en texte libre. Les séances 2 et 10 les reprendront.
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance"]

X = train.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in train.columns])
y = train[CIBLE]

# Les colonnes non numériques deviennent catégorielles. Deux pièges à connaître dès ici :
#   · `is_numeric_dtype` et non `dtype == "object"` — pandas 3 a un dtype `str` dédié ;
#   · une colonne à très forte cardinalité n'est PAS une catégorielle utilisable : les
#     modèles d'arbres plafonnent à 255 modalités. Au-delà, il faut un autre traitement (S10).
CAT = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]

TROP_FIN = [c for c in CAT if X[c].nunique() > 255]
if TROP_FIN:
    print("écartée(s), cardinalité > 255 :", [(c, int(X[c].nunique())) for c in TROP_FIN])
X = X.drop(columns=TROP_FIN)
CAT = [c for c in CAT if c not in TROP_FIN]

for c in CAT:
    X[c] = X[c].fillna("").astype("category")

print("après lecture \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")
print("cible :", round(100 * y.mean(), 2), "% de positifs")
print("colonnes catégorielles :", CAT)

In [ ]:
# 4 — LA RÉFÉRENCE : que vaut un modèle qui répond toujours la même chose ?
# Sans ce chiffre, votre score ne veut rien dire. Notez-le, vous le réutiliserez toute l'année.
reference = 100 * max(y.mean(), 1 - y.mean())

print("la référence, pour tout le semestre \n")
print(f"référence (classe majoritaire) : {reference:.2f} % d'exactitude")

In [ ]:
# 5 — un premier modèle, et une mesure honnête sur des données qu'il n'a pas vues
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0)
modele.fit(Xtr, ytr)

score_interne = 100 * modele.score(Xva, yva)

print("le score interne, à comparer à la référence \n")
print(f"score interne : {score_interne:.2f} %   (référence : {reference:.2f} %)")

In [ ]:
# 6 — prédire sur le test et écrire le fichier de soumission
Xt = test.drop(columns=[c for c in A_ECARTER + TROP_FIN if c in test.columns])

for c in CAT:                       # MÊMES catégories que l'entraînement, sinon décalage silencieux
    Xt[c] = Xt[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)

soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)

print("le fichier écrit : submission.csv \n")
print("dimensions :", soumission.shape, "(lignes, colonnes)")
print("les cinq premières lignes \n")
soumission.head()

### Ce que vous venez de faire — et comment ça s'appelle

- `patient_absent` est **la cible** : ce que le modèle doit deviner.
- Le score de la classe majoritaire est **une référence**. Sans elle, aucun score ne veut
  rien dire.
- Les colonnes écartées « parce qu'on ne les a pas encore vues » sont des **fuites** : elles
  n'existeraient pas au moment où la décision se prend. Développé en S2.
- `tournee_jour` a été écartée : trop de modalités pour un modèle d'arbres, qui plafonne à
  255. C'est **la cardinalité**. Développé en S10.
- La fonction `predict` de la dernière cellule s'appelle **une interface** : on ne relit pas
  un notebook pour exécuter un modèle, on appelle une fonction.

**Aucune de ces notions ne vous est expliquée aujourd'hui.** Elles ont un nom, et ce nom
servira quand la séance les rendra nécessaires.

## Exercice — améliorer le score sans rien casser

Trois choses, dans cet ordre, et **notez les trois scores** au fur et à mesure :

**(a)** remplacez le modèle par un autre (vous en connaissez un ? essayez) ;
**(b)** ajoutez **une feature triviale de votre cru** — une seule, et expliquez-la en une phrase ;
**(c)** consignez vos essais dans un tableau : hypothèse, ce que vous avez changé, score.

**Ce qui compte n'est pas le score, c'est le tableau.** Un essai qui n'a rien amélioré est un
résultat : il vous dit où ne pas chercher. Les séances 9 et 12 vous demanderont une série
d'essais documentés — vous commencez maintenant.

In [ ]:
# 7 — le tableau d'essais (c'est lui qui est lu, pas le score)
ESSAIS = []

def noter(hypothese, changement, score):
    """Ajoute une ligne au tableau des essais."""
    ESSAIS.append({"hypothèse": hypothese, "changement": changement, "score": round(score, 2)})
    print(pd.DataFrame(ESSAIS).to_string(index=False))

noter("rien faire du tout", "classe majoritaire", reference)

print("un premier essai de plus \n")
noter("un modèle suffit-il ?", "HistGradientBoosting", score_interne)

In [ ]:
# 8 — (a) un autre modèle : décommentez, adaptez, relancez la cellule 6
# from sklearn.linear_model import LogisticRegression
# modele = LogisticRegression(max_iter=1000)
# ... réentraînez, remesurez, puis :
# noter("linéaire contre arbres", "LogisticRegression", score_interne)

In [ ]:
# 9 — (b) une feature de votre cru. Une seule. Appliquée DES DEUX CÔTÉS, sinon décalage
# silencieux : le modèle apprend sur une version des données et prédit sur une autre.
def ajouter_features(df):
    """Vos features maison. Appliquée identiquement à l'entraînement et au test."""
    df = df.copy()
    # Exemple — remplacez-le par la vôtre, et dites en une phrase ce qu'elle raconte :
    # df["plus_de_60_ans"] = (df["age"] > 60).astype(int)
    return df

# X  = ajouter_features(X)    puis relancez 5 et 6
# Xt = ajouter_features(Xt)
# noter("une feature de plus", "votre feature", score_interne)

## Une fonction, un test, un dépôt

À partir de maintenant, tout ce que vous rendez doit pouvoir être appelé **de l'extérieur du
notebook** : un module `.py` avec **une fonction, une interface figée**. C'est la même exigence
que dans un vrai poste — on ne relit pas un notebook pour exécuter un modèle.

```python
# modele_equipe.py
def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df."""
```

La cellule suivante définit cette interface et **la teste**. Un test qui échoue vous dit que
vous avez cassé quelque chose — c'est son seul travail.

In [ ]:
# 10 — l'interface imposée, et son test
def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours
    appellera — pas le notebook. Gardez-la vivante en même temps que vous essayez vos idées."""
    d = df.drop(columns=[c for c in A_ECARTER + TROP_FIN if c in df.columns], errors="ignore")

    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)

    return modele.predict(d).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"

    entiers = sum(p)

    assert 0 <= entiers <= 500, "nombre de 1 incohérent"
    print(f"test OK — 500 prédictions, {entiers} positifs")

test_predict()

In [ ]:
# 11 — le dépôt (fait en séance, 10 minutes)
#   git init && git add . && git commit -m "J1" && git remote add origin <votre dépôt>
#   git push -u origin main
# Votre identité n'est à renseigner que si vous committez depuis un terminal (local ou
# Codespaces) : par l'interface web, GitHub signe le commit avec votre compte.
import subprocess

try:
    for cle in ("user.name", "user.email"):
        r = subprocess.run(["git", "config", "--global", cle], capture_output=True, text=True)
        v = r.stdout.strip()
        # On affiche l'ÉTAT, pas la valeur : inutile d'imprimer votre adresse à l'écran,
        # et un notebook forké emporte ses sorties avec lui.
        print(f"  git {cle:<10} : {'renseigné' if v else 'NON RENSEIGNÉ — à faire'}")
except FileNotFoundError:
    print("git n'est pas disponible ici — les commandes ci-dessus sont à passer en local")